## 🧹 Data Preprocessing & Feature Extraction for Resume Classification

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import re
import string
import nltk
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import joblib

In [ ]:
nltk.download('stopwords')

### 2. Load and Clean Dataset

In [ ]:
# Load dataset
path = '../data/resume-dataset/UpdatedResumeDataSet.csv'
DataSet = pd.read_csv(path)

In [ ]:
# Drop duplicates
DataSet.drop_duplicates(subset=['Resume'], inplace=True)
DataSet.reset_index(drop=True, inplace=True)

## 3. Create a Copy for Cleaning

In [ ]:
cleaned_DataSet = DataSet.copy()

## 4. Clean Resume Text

In [ ]:
def cleanResume(resumeText):
    resumeText = re.sub('http\\S+\\s*', ' ', resumeText)
    resumeText = re.sub('RT|cc', ' ', resumeText)
    resumeText = re.sub('#\\S+', '', resumeText)
    resumeText = re.sub('@\\S+', ' ', resumeText)
    resumeText = re.sub('[%s]' % re.escape(string.punctuation), ' ', resumeText)
    resumeText = re.sub(r'[^\x00-\x7f]', r' ', resumeText)
    resumeText = re.sub('\\s+', ' ', resumeText)
    return resumeText

In [ ]:
cleaned_DataSet['Cleaned_Resume'] = cleaned_DataSet['Resume'].apply(cleanResume)

### 5. Encode Target Labels

In [ ]:
le = LabelEncoder()
cleaned_DataSet['Category_encoded'] = le.fit_transform(cleaned_DataSet['Category'])

### 6. Tokenize Text

In [ ]:
stop_words = set(stopwords.words('english') + ['``', "''"])

def tokenize_and_filter(text):
    tokens = word_tokenize(text, preserve_line=True)
    return [word for word in tokens if word.lower() not in stop_words and word not in string.punctuation]

In [ ]:
cleaned_DataSet['Tokens'] = cleaned_DataSet['Cleaned_Resume'].apply(tokenize_and_filter)
cleaned_DataSet['Filtered_Text'] = cleaned_DataSet['Tokens'].apply(lambda tokens: ' '.join(tokens))

### 7. Split and Extract TF-IDF Features

In [ ]:
texts = cleaned_DataSet['Filtered_Text']
labels = cleaned_DataSet['Category_encoded']
train_text, test_text, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)
vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words='english', max_features=1500)
X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)

In [ ]:
print("\n✅ Train shape:", X_train.shape)
print("✅ Test shape:", X_test.shape)

### 8. Save Data with Joblib

In [ ]:
Path('../models').mkdir(exist_ok=True)
joblib.dump(X_train, '../models/X_train.pkl')
joblib.dump(X_test, '../models/X_test.pkl')
joblib.dump(y_train, '../models/y_train.pkl')
joblib.dump(y_test, '../models/y_test.pkl')

# ✅ Data is now preprocessed, vectorized, split, and saved. Ready for model training.